# Validate Clickstream
Validates raw clickstream table for further transformations.


In [0]:
%sql
SELECT *
FROM ecommerce_catalog.bronze.clickstream_raw
ORDER BY event_id;

In [0]:
%sql
DESCRIBE ecommerce_catalog.bronze.clickstream_raw;

In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(DISTINCT event_id) AS unique_events,
  SUM(CASE WHEN browser IS NULL THEN 1 ELSE 0 END) AS missing_browsers,
  SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) AS missing_countries,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customers,
  SUM(CASE WHEN device_type IS NULL THEN 1 ELSE 0 END) AS missing_device_types,
  SUM(CASE WHEN event_id IS NULL THEN 1 ELSE 0 END) AS missing_event_id,
  SUM(CASE WHEN event_timestamp IS NULL THEN 1 ELSE 0 END) AS missing_event_timestamp,
  SUM(CASE WHEN event_type IS NULL THEN 1 ELSE 0 END) AS missing_event_type,
  SUM(CASE WHEN page_url IS NULL THEN 1 ELSE 0 END) AS missing_page_url,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS missing_product_id,
  SUM(CASE WHEN session_id IS NULL THEN 1 ELSE 0 END) AS missing_session_id,
  SUM(CASE WHEN traffic_source IS NULL THEN 1 ELSE 0 END) AS missing_traffic_source
FROM ecommerce_catalog.bronze.clickstream_raw;

In [0]:
%sql
SELECT event_id, count(*)
FROM ecommerce_catalog.bronze.clickstream_raw
GROUP BY event_id
HAVING count(*) > 1;

In [0]:
%sql
SELECT *
FROM ecommerce_catalog.bronze.clickstream_raw
WHERE event_id = "evt_000805";

In [0]:
%sql
SELECT device_type
FROM ecommerce_catalog.bronze.clickstream_raw
ORDER BY device_type;

In [0]:
%sql
SELECT event_type
FROM ecommerce_catalog.bronze.clickstream_raw
ORDER BY event_type;

In [0]:
%sql
SELECT event_timestamp
FROM ecommerce_catalog.bronze.clickstream_raw
ORDER BY event_timestamp DESC;

In [0]:
%sql
SELECT event_timestamp
FROM ecommerce_catalog.bronze.clickstream_raw
WHERE event_timestamp LIKE "%T%";

### **Observations:**
- The `event_timestamp` column is stored as `STRING` in the Bronze layer.
- The majority of timestamps use the `YYYY-MM-DD HH:mm:ss` format.
- 15 timestamps alternatively use the valid `YYYY-MM-DDTHH:mm:ss` format in `ecommerce_catalog.bronze.clickstream_raw`.
- One record contains an invalid timestamp: `INVALID_TIME_STAMP`.
- 187 `event_id` values appear multiple times. Affected records are being evaluated for exact duplicates.
- The following `NULL` counts were identified:
    - `customer_id`: 126
    - `product_id`: 123
    - `browser`: 3
    - `traffic_source`: 1
- Missing `customer_id` values do not necessarily indicate data corruption, as anonymous guest sessions are valid in clickstream data.
- Missing `product_id` values must be evaluated in relation to `event_type`: they are expected for generic page visits, but require further validation for product-specific events.
- Inconsistent casing was identified in `device_type`.
- Inconsistent casing was identified in `event_type`.

### **Silver Layer Actions:**
- Cast `event_timestamp` to a unified timestamp datatype, handling both valid source formats.
- Treat `INVALID_TIME_STAMP` as a missing/invalid timestamp, as it cannot be reliably recovered.
- Remove verified exact duplicate events.
- Standardize `device_type` and `event_type` to consistent casing.
- Evaluate missing `customer_id` and `product_id` values within their business context rather than applying blanket imputation or row deletion.
- Impute missing `browser` and `traffic_source` values only where reliable reconstruction is possible; otherwise, retain them as `NULL`.